## Chuẩn bị tên 25 vị trí

In [5]:
import torch
import torch.nn as nn
import pandas as pd

conditions = [
    "spinal_canal_stenosis",
    "left_neural_foraminal_narrowing",
    "right_neural_foraminal_narrowing",
    "left_subarticular_stenosis",
    "right_subarticular_stenosis",
]
levels = ["l1_l2", "l2_l3", "l3_l4", "l4_l5", "l5_s1"]
class_names = ["Normal/Mild", "Moderate", "Severe"]

target_names = [
    f"{condition}_{level}"
    for condition in conditions
    for level in levels
]

print("Số vị trí:", len(target_names))  # 25

Số vị trí: 25


## Xây dựng CNN phân loại một crop

In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, stride=1, padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.block(x)


class LumbarCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 3),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)  # logits, chưa qua Softmax

## Chạy 25 crop qua CNN và dùng Softmax

In [7]:
torch.manual_seed(42)

# Ảnh giả để chạy thử; về sau thay bằng 25 crop MRI thật.
images = torch.randn(1, 25, 1, 64, 64)

batch_size, num_targets, channels, height, width = images.shape

model = LumbarCNN()
model.eval()

with torch.inference_mode():
    # Gộp 1 ca × 25 crop thành 25 ảnh để đưa vào CNN.
    flat_images = images.reshape(
        batch_size * num_targets, channels, height, width
    )

    flat_logits = model(flat_images)  # [25, 3]

    # Ghép kết quả về: mỗi ca có 25 vị trí, mỗi vị trí có 3 logits.
    logits = flat_logits.reshape(batch_size, num_targets, 3)  # [1, 25, 3]

    # Chuyển 3 logits của từng vị trí thành 3 xác suất.
    probabilities = torch.softmax(logits, dim=-1)

print("Ảnh đầu vào:", tuple(images.shape))
print("Logits:", tuple(logits.shape))
print("Xác suất:", tuple(probabilities.shape))
print("Tổng xác suất vị trí đầu:", probabilities[0, 0].sum().item())

Ảnh đầu vào: (1, 25, 1, 64, 64)
Logits: (1, 25, 3)
Xác suất: (1, 25, 3)
Tổng xác suất vị trí đầu: 1.0


## Xem kết quả dưới dạng bảng

In [ ]:
rows = []

for target_name, probs in zip(target_names, probabilities[0]):
    predicted_class = probs.argmax().item()

    rows.append({
        "condition_level": target_name,
        "Normal/Mild": probs[0].item(),
        "Moderate": probs[1].item(),
        "Severe": probs[2].item(),
        "dự đoán": class_names[predicted_class],
    })

result = pd.DataFrame(rows)
display(result.round(4))

print("Số dòng:", len(result))  # 25

,condition_level,Normal/Mild,Moderate,Severe,dự đoán
0,spinal_canal_stenosis_l1_l2,0.3176,0.3614,0.3210,Moderate
1,spinal_canal_stenosis_l2_l3,0.3176,0.3614,0.3211,Moderate
2,spinal_canal_stenosis_l3_l4,0.3174,0.3618,0.3208,Moderate
3,spinal_canal_stenosis_l4_l5,0.3174,0.3616,0.3210,Moderate
4,spinal_canal_stenosis_l5_s1,0.3176,0.3613,0.3211,Moderate
5,left_neural_foraminal_narrowing_l1_l2,0.3172,0.3617,0.3211,Moderate
6,left_neural_foraminal_narrowing_l2_l3,0.3174,0.3617,0.3209,Moderate
7,left_neural_foraminal_narrowing_l3_l4,0.3175,0.3613,0.3212,Moderate
8,left_neural_foraminal_narrowing_l4_l5,0.3172,0.3621,0.3208,Moderate
9,left_neural_foraminal_narrowing_l5_s1,0.3176,0.3613,0.3211,Moderate


Số dòng: 25
